# FBA Analysis

This code runs a FBA Analysis and provides a biomass_values.csv and a viability_matrix.csv

In [1]:
import os
import pandas as pd
import cobra
from cobra.io import read_sbml_model, write_sbml_model

In [6]:
# ======== CONFIGURATION ========
input_folder = "/scratch/rebernig/VR002_Model_Analysis/Output/Curated_Models_Elena/P.Vulgatus/Confidence_Levels_Biomass"
output_folder = "/scratch/rebernig/VR002_Model_Analysis/Output/Curated_Models_Elena/P.Vulgatus"
biomass_csv = "/scratch/rebernig/VR002_Model_Analysis/Output/Curated_Models_Elena/P.Vulgatus/Biomass_Viability/biomass_values.csv"
viability_csv = "/scratch/rebernig/VR002_Model_Analysis/Output/Curated_Models_Elena/P.Vulgatus/Biomass_Viability/viability_matrix.csv"
media_csv = "/scratch/rebernig/VR002_Model_Analysis/Scripts/Elenas_models/Transcriptomics_integration/Media_LB_minus_02/LB_minus_O2_media.csv"
save_modified_models = False  # Set to False if you don't want to save the modified SBMLs
# Ensure output folder exists
os.makedirs(output_folder, exist_ok=True)

In [7]:
# ======== VB MEDIUM DEFINITION ========
# --- load media ONCE ---
LB_minus_O2_media = pd.read_csv(media_csv, index_col=0).squeeze().to_dict()  # {met_or_met_e: cap}

def apply_media_by_metabolite_map(model, media_map, anaerobic=True, atpm_lb=5.54):
    """Apply medium given as {met_id or met_id_e: uptake_cap} to a COBRA model."""
    med_mod = {ex.id: 0.0 for ex in model.exchanges}  # start closed
    for ex in model.exchanges:
        met = next(iter(ex.metabolites.keys()))
        met_id = met.id                       # e.g. "glc__D_e"
        base_id = met_id[:-2] if met_id.endswith("_e") else met_id  # "glc__D"
        cap = media_map.get(met_id, media_map.get(base_id, 0.0))
        med_mod[ex.id] = float(cap)
    if anaerobic and "EX_o2_e" in med_mod:
        med_mod["EX_o2_e"] = 0.0
    model.medium = med_mod
    if atpm_lb is not None and "ATPM" in model.reactions:
        model.reactions.ATPM.lower_bound = float(atpm_lb)

df_biomass = {}

In [8]:
# --- FBA over models ---
for file_name in sorted(os.listdir(input_folder)):
    if not file_name.endswith(".xml"):
        continue
    file_path = os.path.join(input_folder, file_name)
    print(f"\n🔧 Processing: {file_name}")
    try:
        model = read_sbml_model(file_path)

        # >>> APPLY MEDIUM HERE <<<
        apply_media_by_metabolite_map(model, LB_minus_O2_media, anaerobic=True, atpm_lb=5.54)

        solution = model.optimize()
        biomass = solution.objective_value if solution.status == "optimal" else None

        # Clamp negatives to zero
        if biomass is not None and biomass < 0:
            biomass = 0.0
        # (optional) use a tolerance instead:
        # if biomass is not None and biomass < 1e-9: biomass = 0.0

        df_biomass[os.path.splitext(file_name)[0]] = biomass
    except Exception as e:
        print(f"❌ Error in model {file_name}: {e}")
        df_biomass[os.path.splitext(file_name)[0]] = None




🔧 Processing: agora.xml

🔧 Processing: assembly.xml

🔧 Processing: carveme.xml

🔧 Processing: core2.xml

🔧 Processing: core3.xml

🔧 Processing: core4.xml

🔧 Processing: core5.xml


/home/rebernig/.conda/envs/gemsembler-test/lib/python3.10/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)



🔧 Processing: curated.xml

🔧 Processing: gapseq.xml

🔧 Processing: kbase.xml

🔧 Processing: modelseed.xml


In [9]:
ser_biomass = pd.Series(df_biomass)
ser_viable = ser_biomass.apply(lambda x: 1 if x is not None and x > 1e-6 else 0)
ser_biomass.to_csv(biomass_csv)
ser_viable.to_csv(viability_csv)
print("\n✅ Finished processing all models.")
print(f"📄 Biomass values saved to: {biomass_csv}")
print(f"📄 Viability matrix saved to: {viability_csv}")


✅ Finished processing all models.
📄 Biomass values saved to: /scratch/rebernig/VR002_Model_Analysis/Output/Curated_Models_Elena/P.Vulgatus/Biomass_Viability/biomass_values.csv
📄 Viability matrix saved to: /scratch/rebernig/VR002_Model_Analysis/Output/Curated_Models_Elena/P.Vulgatus/Biomass_Viability/viability_matrix.csv


In [16]:
flux = solution.fluxes
print(flux)

LCADi       0.000000
G6PI3      -2.069300
GHMT2r      1.259538
GCALDt      0.000000
ACONT       0.000000
              ...   
SERabc      0.000000
G3PAT180    0.058157
PGSA181     0.014539
PGPP181     0.014539
Biomass     1.160309
Name: fluxes, Length: 1018, dtype: float64
